# 第 7 章: cinema データの探索と可視化

特徴量と興行収入の関係、外れ値、線形回帰の予測誤差を確かめる。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: FSharp.Stats, 0.6.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter02.IrisPreprocessing
open MachineLearning.Chapter07.Cinema
open MachineLearning.Chapter07.LinearRegression
open MachineLearning.Chapter07.RegressionMetrics

let cinemaCsv = Path.Combine(dataDir (), "cinema.csv")
let rows = loadCinema cinemaCsv
rows |> List.map (fun row -> row.Features) |> countMissing

## 相関を見る

In [ ]:
let featureNames = [ "SNS1"; "SNS2"; "actor"; "original" ]

// 欠損値のある行を除いてから、特徴量と興行収入の相関係数を求める
let complete =
    rows |> List.filter (fun row -> row.Features |> Map.forall (fun _ value -> value.IsSome))

let column (name: string) =
    complete |> List.map (fun row -> row.Features[name].Value)

let sales = complete |> List.map (fun row -> row.Sales)

featureNames
|> List.map (fun name -> {| 特徴量 = name; 相関係数 = FSharp.Stats.Correlation.Seq.pearson (column name) sales |})
|> List.sortByDescending (fun row -> row.相関係数)
|> List.toArray

In [ ]:
// 特徴量同士の相関係数
[| for a in featureNames ->
       {| 特徴量 = a
          SNS1 = FSharp.Stats.Correlation.Seq.pearson (column a) (column "SNS1")
          SNS2 = FSharp.Stats.Correlation.Seq.pearson (column a) (column "SNS2")
          actor = FSharp.Stats.Correlation.Seq.pearson (column a) (column "actor")
          original = FSharp.Stats.Correlation.Seq.pearson (column a) (column "original") |} |]

## 散布図で外れ値を確かめる

In [ ]:
let kept = removeOutliers rows |> List.map (fun row -> row.CinemaId) |> Set.ofList
let outliers, normal = rows |> List.partition (fun row -> not (kept.Contains row.CinemaId))

let points (group: CinemaRow list) (name: string) =
    group
    |> List.choose (fun row -> row.Features["SNS2"] |> Option.map (fun sns2 -> sns2, row.Sales))
    |> fun xy -> Chart.Point(xy = xy, Name = name)

[ points normal "外れ値ではない"; points outliers "外れ値" ]
|> Chart.combine
|> Chart.withTitle "SNS2 と興行収入"
|> Chart.withXAxisStyle "SNS2"
|> Chart.withYAxisStyle "sales"

## 実測値と予測値、残差

In [ ]:
let split = prepareCinema cinemaCsv 0.2 0
let model = fitLinearRegression split.XTrain split.TTrain
let y = predictLinearRegression model split.XTest
let residuals = List.map2 (-) split.TTest y

Chart.Point(x = split.TTest, y = y)
|> Chart.withTitle "テストデータの実測値と予測値"
|> Chart.withXAxisStyle "実測値"
|> Chart.withYAxisStyle "予測値"

In [ ]:
Chart.Point(x = y, y = residuals)
|> Chart.withTitle "予測値と残差（実測値 - 予測値）"
|> Chart.withXAxisStyle "予測値"
|> Chart.withYAxisStyle "残差"

In [ ]:
{| 最小 = List.min residuals; 最大 = List.max residuals |}

## 外れ値を除く効果を、同じテストデータで比べる

In [ ]:
/// 訓練データの平均で補完して学習し、SNS2 の係数とテストデータの決定係数を返す
let evaluate (train: CinemaRow list) (test: CinemaRow list) =
    let means = train |> List.map (fun row -> row.Features) |> columnMeans
    let fill (group: CinemaRow list) = group |> List.map (fun row -> row.Features) |> fillMissing means
    let fitted = fitLinearRegression (fill train) (train |> List.map (fun row -> row.Sales))
    let predicted = predictLinearRegression fitted (fill test)
    fitted.Coefficients["SNS2"], r2Score (test |> List.map (fun row -> row.Sales)) predicted

// 行ごと分割するので、x と t の両方に行そのものを渡す
let sameSplit = splitTrainTest 0.2 0 rows rows

[|
    let coefficient, r2 = evaluate sameSplit.XTrain sameSplit.XTest
    {| 学習 = "外れ値を残して学習"; SNS2の係数 = coefficient; テストデータのR2 = r2 |}
    let coefficient, r2 = evaluate (removeOutliers sameSplit.XTrain) sameSplit.XTest
    {| 学習 = "外れ値を除いて学習"; SNS2の係数 = coefficient; テストデータのR2 = r2 |}
|]

## 分け方によって R2 が変わる

In [ ]:
[| for seed in 0..4 ->
       let s = prepareCinema cinemaCsv 0.2 seed
       {| シード = seed
          テストデータのR2 = r2Score s.TTest (predictLinearRegression (fitLinearRegression s.XTrain s.TTrain) s.XTest) |} |]